# Notebook 27: Self-Attention - Building Attention from Scratch for Images

---

## What This Notebook Covers

This notebook builds the **self-attention** mechanism from first principles and adapts it to work on 2D convolutional feature maps — the exact form of attention that later gets bolted into the diffusion U-Net to let it model *global* relationships across an image. We will learn:

1. **From feature maps to sequences** — how a `(batch, channels, height, width)` activation is reshaped into a `(batch, sequence, channels)` tensor so that attention can treat every spatial location as a "token"
2. **Query, Key, Value projections** — the three learned linear maps at the heart of attention, first built by hand
3. **Scaled dot-product attention** — the core equation `softmax(QKᵀ/√d) V`, derived and motivated piece by piece
4. **A `SelfAttention` module** — packaging the mechanism into a residual `nn.Module` with normalization
5. **Validating against 🤗 diffusers** — copying our weights into HuggingFace's `AttentionBlock` to prove our implementation is numerically identical
6. **The combined QKV projection** — a small efficiency refactor (`nn.Linear(ni, ni*3)` + `chunk`)
7. **Multi-head attention** — splitting the channel dimension into several independent attention "heads", including the `einops` `rearrange` trick that makes it a one-liner
8. **Comparison to `nn.MultiheadAttention`** — checking our from-scratch multi-head module against PyTorch's built-in

---

## Why Attention?

A convolution is **local**: each output pixel is a weighted sum of a small neighborhood (3×3, 5×5) of the input. To let information travel across the whole image, a CNN has to stack many conv layers so the *receptive field* slowly grows. That is fine for texture and edges, but it is an awkward way to express a relationship like *"this patch of sky at the top-left should be consistent with that patch of sky at the top-right"* — those pixels are far apart, and many layers of convolution stand between them.

**Attention** is the opposite: it is **global by construction**. Every location gets to look at *every other location* in a single operation and decide, per-location, how much to borrow from each. The "how much" is computed dynamically from the content itself (not fixed weights like a conv kernel), so the same layer can attend to the top-left↔top-right sky in one image and to a foreground↔background relationship in the next.

This is why attention became the backbone of Transformers in NLP, and why it is now a standard ingredient in image generators: in a diffusion U-Net, the deepest (lowest-resolution) feature maps are exactly where a global "does this whole image hang together?" operation is most valuable, and self-attention supplies it.

**A climate/EO bridge (a real one).** The attention score matrix — an `S × S` matrix saying how strongly location *i* should attend to location *j* — is structurally identical to a **teleconnection / spatial-covariance matrix**: a data-driven map of which locations in a field are dynamically coupled to which others (think ENSO linking the tropical Pacific to remote weather). A convolution can only express *local* spatial coupling; attention learns *long-range* coupling directly, the same way an EOF/teleconnection analysis surfaces remote correlations that a local stencil never could. Keep this analogy in your back pocket — the `q@k.transpose` line below is literally building that coupling matrix.

---

## Prerequisites

You should be comfortable with:

- **Convolutional feature maps** and their `(N, C, H, W)` layout (notebooks 07–08)
- **`nn.Linear`** as a learned affine map applied to the last dimension of a tensor (notebook 03 onward)
- **Batch/Group normalization** (notebooks 10–11) — we use `GroupNorm`/`BatchNorm2d` as the pre-norm here
- **Matrix multiplication and broadcasting semantics** (notebook 01), especially batched matmul (`@` on 3D tensors)
- **Softmax** as a row-wise normalizer that turns scores into a probability distribution

---


# Part 1: Setup and Imports

We need very little: PyTorch itself, our `miniai` helpers (for `set_seed`), matplotlib, and — crucially — HuggingFace's `diffusers` library, whose `AttentionBlock` we will use later as a **reference implementation** to check our work against.

---


In [ ]:
# =====================================================
# CORE IMPORTS
# =====================================================
import math,torch                    # math.sqrt for the attention scale factor; torch for tensors
from torch import nn                  # nn.Linear, nn.Module, normalization layers
from miniai.activations import *      # brings in set_seed (and other helpers) from our library

**What does the code above do?**

Standard setup. The only non-obvious import is `from miniai.activations import *`, which pulls in `set_seed` (the reproducibility helper we built earlier) along with the rest of the `activations` module. `math` is here purely for `math.sqrt`, which will be the `√d` scaling constant inside attention.


In [ ]:
import matplotlib.pyplot as plt    # imported for convenience; not heavily used in this notebook

**What does the code above do?**

Pulls in matplotlib. This notebook is almost entirely about tensor *shapes* and *values*, so there is very little plotting — but it is conventional to import it in these lessons.


In [ ]:
from diffusers.models.attention import AttentionBlock   # HuggingFace's reference self-attention block

**What does the code above do?**

Imports `AttentionBlock` from the 🤗 `diffusers` library. This is the *same* self-attention operation we are about to build by hand, as implemented by the people who ship production diffusion models. Later we will copy our learned weights into it and confirm that our output matches theirs **element for element** — the single most convincing way to prove a from-scratch implementation is correct.

> **Version note.** `diffusers.models.attention.AttentionBlock` exists in the `diffusers` version used in the course (circa 2023). In newer `diffusers` releases this class was renamed/relocated (e.g. `Attention` / `AttnBlock` in `attention_processor`). If this import fails on your install, that is a library-version issue, not a problem with the attention *concept* — everything else in the notebook is pure PyTorch and will run regardless.


---

# Part 2: From Feature Maps to Sequences

Attention was invented for **sequences** (sentences: a sequence of word tokens). Its native input shape is `(batch, sequence_length, features)` — a batch of variable-length lists of feature vectors.

But a convolutional feature map is `(batch, channels, height, width)` — a 2D grid, not a 1D list. Before we can apply attention to an image, we have to **reinterpret the spatial grid as a sequence**: flatten the `H×W` grid into a single sequence of `H·W` positions, and treat the `C` channels at each position as that position's feature vector.

Let's set up a dummy activation tensor and do exactly that reshape.

---


In [ ]:
# =====================================================
# A DUMMY CONVOLUTIONAL ACTIVATION
# =====================================================
set_seed(42)                     # reproducibility: same random tensor every run
x = torch.randn(64,32,16,16)     # (N=64 images, C=32 channels, H=16, W=16) — a typical deep feature map

**What does the code above do?**

Creates a fake activation `x` shaped like something that would come out of a conv layer deep in a network:

| Dim | Size | Meaning |
|-----|------|---------|
| N | 64 | batch — 64 images at once |
| C | 32 | channels — 32 feature detectors per pixel |
| H | 16 | feature-map height |
| W | 16 | feature-map width |

So each of the 64 images is a 16×16 grid, and at every one of those 256 grid cells we have a 32-dimensional feature vector. Those 256 cells are what we will turn into a "sequence of 256 tokens, each 32-dimensional."


In [ ]:
# =====================================================
# FLATTEN THE SPATIAL GRID INTO A SEQUENCE
# =====================================================
# x.shape[:2] is (N, C) = (64, 32). The -1 flattens H,W -> H*W = 256.
#   x.view(64, 32, -1)      -> (64, 32, 256)   : channels, then the 256 flattened positions
#   .transpose(1, 2)        -> (64, 256, 32)   : positions, then channels  == (batch, seq, feats)
t = x.view(*x.shape[:2], -1).transpose(1, 2)
t.shape

torch.Size([64, 256, 32])

**What does the code above do?**

This is the single most important *shape* move in the whole notebook. Read it in two steps:

1. **`x.view(64, 32, -1)`** collapses the last two axes (`H=16`, `W=16`) into one axis of length `16·16 = 256`. Shape becomes `(64, 32, 256)` — "for each image, for each of 32 channels, a length-256 vector of spatial responses." No data is copied or moved; `view` just relabels how the flat memory is indexed.

2. **`.transpose(1, 2)`** swaps the channel axis and the position axis. Shape becomes `(64, 256, 32)` — "for each image, for each of 256 spatial positions, a 32-dimensional feature vector."

That final layout, `(batch=64, seq_len=256, features=32)`, is **exactly** what attention expects. We have reframed the image as a sentence of 256 "words," where each word is the 32-channel feature vector living at one pixel.

```
   (N, C, H, W)              view              (N, C, H*W)          transpose(1,2)        (N,  S,  C)
  (64,32,16,16)  ───────────────────────▶   (64, 32, 256)   ───────────────────────▶  (64, 256, 32)
   image grid                                channels ×                                 sequence of
                                             flat positions                             256 feature vectors
```

**Why does this ordering matter?** `nn.Linear` and the attention matmuls all act on the **last** dimension and treat everything before it as "batch". By putting `features=32` last and `seq=256` second, a single `nn.Linear(32, 32)` transforms every one of the 256 positions independently, and a batched matmul over the `seq` axis lets every position interact with every other. If we had left channels last-but-one, none of the downstream code would line up.


## Deep Dive: Why "flatten to a sequence" is the whole trick

New readers often stumble here because the reshape looks like bookkeeping, when in fact it *is* the conceptual leap. Let's slow down.

### The problem attention solves, restated for images

Convolution answers: *"What is here, given the small neighborhood around here?"* Its weights are **fixed** after training and **local** in reach.

Attention answers: *"For this location, which other locations (anywhere in the image) are relevant, and by how much — decided right now from the actual content?"* Its "weights" (the attention scores) are **computed on the fly** and **global** in reach.

To even *ask* that question, we need every location to be an addressable item that can be compared against every other location. A flat sequence of feature vectors is precisely such an addressable collection: position *i* is one vector, position *j* is another, and "how relevant is *j* to *i*" is a number we get by comparing those two vectors.

### Why the feature vector must be the last axis

Everything downstream is a matmul or a `Linear`, and PyTorch's convention is: **operate on the last axis, batch over the rest.** Concretely, with `t` shaped `(64, 256, 32)`:

- `nn.Linear(32, 32)(t)` applies the same 32→32 map to each of the `64×256` feature vectors independently. That is what we want: the Q/K/V projections should treat each position the same.
- `t @ t.transpose(1,2)` does a **batched** matmul: for each of the 64 images independently, it multiplies a `(256, 32)` matrix by a `(32, 256)` matrix to get a `(256, 256)` matrix — the all-pairs comparison of positions. That `256×256` object is the coupling matrix we cared about.

### The reshape is fully reversible

Nothing is lost. After attention has mixed information across positions, we run the same two moves in reverse — `transpose(1,2)` then `reshape(N, C, H, W)` — to snap the sequence back into an image grid so the next conv layer can consume it. You will see exactly that at the end of every `SelfAttention.forward` below. Keep an eye out: **flatten → attend → un-flatten** is the skeleton of every image-attention module.

> **Foreshadowing (this bites us later).** One of the two `SelfAttention` rewrites near the end of this notebook *omits* the flatten step and tries to run `nn.Linear` on the wrong axis. It raises a shape error. That "bug" is the best possible evidence that this reshape is not optional bookkeeping — it is load-bearing.


---

# Part 3: Query, Key, and Value — By Hand

Attention introduces three learned linear projections of the input, traditionally called **Query (Q)**, **Key (K)**, and **Value (V)**. Before wrapping them in a class, Jeremy builds them one at a time so the shapes are concrete. Let's do the same.

---


In [ ]:
ni = 32     # ni = "number of inputs" = the feature/channel dimension (32 here)

**What does the code above do?**

Names the feature dimension `ni = 32` (matching the channel count of `x`). In this notebook the Q/K/V projections are all `32 → 32`, so input and output feature sizes are equal. `ni` is just shorthand we'll reuse when constructing the `nn.Linear` layers.


In [ ]:
# =====================================================
# THREE INDEPENDENT LINEAR PROJECTIONS: KEY, QUERY, VALUE
# =====================================================
sk = nn.Linear(ni, ni)   # sk: produces the KEYS    (what each position "offers" as a match target)
sq = nn.Linear(ni, ni)   # sq: produces the QUERIES (what each position "is looking for")
sv = nn.Linear(ni, ni)   # sv: produces the VALUES  (what each position "contributes" if attended to)

**What does the code above do?**

Creates three *separate* `nn.Linear(32, 32)` layers — one each for keys, queries, and values. Each has its own weights, so from the same input feature vector the network can compute three different 32-d vectors:

| Projection | Intuition (the "search" metaphor) |
|-----------|------------------------------------|
| **Query** `q = sq(t)` | For position *i*: *"here is what I'm looking for."* |
| **Key** `k = sk(t)` | For position *j*: *"here is what I can be matched on."* |
| **Value** `v = sv(t)` | For position *j*: *"here is the content I'll hand over if you attend to me."* |

The comparison `query·key` measures how well what *i* is looking for matches what *j* offers. High match ⇒ *i* pulls a lot of *j*'s **value** into its output. Note the deliberate asymmetry: keys and queries decide *routing*; values carry the *payload*. Splitting "what you match on" from "what you deliver" is exactly what gives attention its expressive power.


In [ ]:
# =====================================================
# APPLY THE PROJECTIONS TO THE SEQUENCE t : (64, 256, 32)
# =====================================================
# Each Linear maps the last dim 32 -> 32, independently at all 64*256 positions.
k = sk(t)    # keys    : (64, 256, 32)
q = sq(t)    # queries : (64, 256, 32)
v = sv(t)    # values  : (64, 256, 32)

**What does the code above do?**

Runs the three projections on our sequence `t` of shape `(64, 256, 32)`. Because `nn.Linear` acts on the last axis and batches over the rest, each of `q`, `k`, `v` comes back the same shape as `t`: `(64, 256, 32)`. We now have, for every one of the 256 positions in every one of the 64 images, a query vector, a key vector, and a value vector.


In [ ]:
# =====================================================
# THE ATTENTION SCORE MATRIX: EVERY QUERY DOTTED WITH EVERY KEY
# =====================================================
# q            : (64, 256, 32)
# k.transpose(1,2): (64, 32, 256)   <- swap seq & feature so the shared dim (32) faces inward
# batched matmul -> (64, 256, 256)  : for each image, an all-pairs score matrix
(q@k.transpose(1,2)).shape

torch.Size([64, 256, 256])

**What does the code above do?**

This is the heart of attention: **compare every query with every key.**

- `q` is `(64, 256, 32)`.
- `k.transpose(1,2)` reshapes the keys to `(64, 32, 256)`, putting the shared feature dimension (32) on the inside so the matmul can contract over it.
- The batched matmul `q @ k.transpose(1,2)` therefore produces `(64, 256, 256)`.

Entry `[b, i, j]` of that result is the **dot product of query *i* with key *j*** in image *b* — a single scalar saying "how much should position *i* attend to position *j*." The full `256×256` matrix is the all-pairs **coupling / affinity matrix** for the image (the teleconnection-matrix analogy from the intro). Every row will next be softmax-normalized into a set of attention weights that sum to 1.


## Deep Dive: Scaled Dot-Product Attention, Derived

We now have all the pieces to write attention's defining equation. It is worth deriving each factor, because Jeremy's code compresses it into three lines and the *why* is easy to miss.

The equation is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d}}\right) V$$

Let's build it up term by term, with shapes for our running example (`S = 256` positions, `d = 32` features, per image).

---

### Step 1 — `Q Kᵀ` : the raw affinity scores

$$S_{ij} = q_i \cdot k_j = \sum_{c=1}^{d} q_{ic}\, k_{jc}$$

For each pair of positions *(i, j)* we take the **dot product** of query *i* and key *j*. A dot product is large and positive when the two vectors point the same way — i.e. when what *i* is looking for aligns with what *j* offers. The result `S` is the `(S × S)` = `(256 × 256)` score matrix (per image). In code this is the `q@k.transpose(1,2)` we just ran.

**Why a dot product specifically?** It is the cheapest possible content-based similarity: a single batched matmul, fully parallel over all pairs, no learned parameters of its own (the learning lives in the Q and K projections). Compare this to a convolution, whose "similarity" is fixed weights against fixed offsets — here the similarity is between *learned representations of the actual content*, recomputed for every input.

---

### Step 2 — divide by `√d` : the scaling that keeps softmax sane

$$\tilde{S}_{ij} = \frac{q_i \cdot k_j}{\sqrt{d}}$$

Here is the derivation of the `√d`. Suppose the entries of `q` and `k` are roughly independent with mean 0 and variance 1. Their dot product is a sum of `d` such products:

$$q_i \cdot k_j = \sum_{c=1}^{d} q_{ic} k_{jc}.$$

Each term `q_{ic} k_{jc}` has mean 0 and variance ≈ 1, and there are `d` independent terms, so the **variance of the sum is ≈ d**, i.e. the standard deviation grows like **√d**. Without correction, the scores fed into softmax have magnitude ∝ √d. For `d = 32` that is already ≈ 5.7×; in a real transformer with `d = 64` or larger it is bigger still.

**Why that is bad:** softmax of large-magnitude logits saturates — it becomes nearly one-hot (all the weight on the single largest score) and its gradient collapses toward zero. Training stalls. Dividing by `√d` **renormalizes the scores back to unit scale** regardless of `d`, keeping softmax in its responsive, well-gradiented regime. In the code this is `/ self.scale` with `self.scale = math.sqrt(ni)`.

> Note on our small example: `ni = 32`, so `scale = √32 ≈ 5.657`. In the multi-head version the scale becomes `√(ni/nheads)` because each head operates on a *smaller* per-head feature dimension — more on that in the multi-head deep dive.

---

### Step 3 — `softmax(·)` over the last axis : scores → attention weights

$$A_{ij} = \frac{\exp(\tilde{S}_{ij})}{\sum_{j'} \exp(\tilde{S}_{ij'})}$$

Softmax is applied **along the last dimension** (`dim=-1`), i.e. **across keys, within each query's row**. This turns each row of the score matrix into a probability distribution: `A_{i,:}` is non-negative and sums to 1. So `A_{ij}` is literally *"the fraction of attention position i pays to position j."*

**Why row-wise (`dim=-1`) and not column-wise?** Because we are building position *i*'s output as a weighted average of everyone else's values, and a weighted average needs weights that sum to 1 **over the things being averaged (the keys/values, the last axis).** Softmaxing the wrong axis would normalize over queries instead and produce nonsense. This is the single most common attention bug; the code guards against it with the explicit `dim=-1`.

---

### Step 4 — `A V` : mix the values

$$\text{out}_i = \sum_{j} A_{ij}\, v_j$$

Finally, multiply the attention-weight matrix `A` (shape `256 × 256`) by the value matrix `V` (shape `256 × 32`) to get the output (shape `256 × 32`). Row *i* of the output is the **attention-weighted sum of all value vectors** — position *i*'s new feature vector, assembled by pulling in content from wherever it decided to look. In code: `s.softmax(dim=-1) @ v`.

---

### Putting it together (the three lines you'll see below)

```python
s = (q @ k.transpose(1,2)) / self.scale   # Steps 1-2: scaled scores      (N, S, S)
x = s.softmax(dim=-1) @ v                  # Steps 3-4: weights, then mix  (N, S, d)
```

That is the entire mechanism. Everything else in a `SelfAttention` module — normalization, the output projection, the residual add, the reshaping — is *packaging* around these two lines.

**One more design note — the output projection.** After mixing values, real attention modules apply one more `nn.Linear` (`self.proj`) before returning. Why? The mixed output is a convex combination of `V` vectors living in "value space"; the `proj` gives the network a learned chance to map that back into the representation space the rest of the network expects (and, in multi-head, to blend information *across* heads). It is cheap and it matters.


---

# Part 4: Packaging It — the `SelfAttention` Module (v1)

Now we wrap the mechanism into a proper `nn.Module`. This first version keeps Q, K, V as three separate `nn.Linear` layers (matching the by-hand code above), adds a **`GroupNorm` pre-normalization**, an **output projection**, and a **residual connection**.

---


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, ni):
        super().__init__()
        self.scale = math.sqrt(ni)       # the 1/sqrt(d) scaling constant from the deep dive
        self.norm = nn.GroupNorm(1, ni)  # pre-norm: GroupNorm with 1 group == LayerNorm over C,H,W
        self.q = nn.Linear(ni, ni)       # query projection
        self.k = nn.Linear(ni, ni)       # key   projection
        self.v = nn.Linear(ni, ni)       # value projection
        self.proj = nn.Linear(ni, ni)    # output projection (post-mix remap)

    def forward(self, x):
        inp = x                          # stash the input for the residual add at the end
        n,c,h,w = x.shape                # remember the image shape so we can restore it later
        x = self.norm(x)                 # normalize BEFORE flattening (still (n,c,h,w))
        x = x.view(n, c, -1).transpose(1, 2)   # (n,c,h,w) -> (n, c, h*w) -> (n, seq, c)  == (n, S, C)
        q = self.q(x)                    # (n, S, C)
        k = self.k(x)                    # (n, S, C)
        v = self.v(x)                    # (n, S, C)
        s = (q@k.transpose(1,2))/self.scale    # scaled scores : (n, S, S)
        x = s.softmax(dim=-1)@v          # attention weights @ values : (n, S, C)
        x = self.proj(x)                 # output projection : (n, S, C)
        x = x.transpose(1,2).reshape(n,c,h,w)  # un-flatten back to image grid : (n, c, h, w)
        return x+inp                     # residual connection

**What does the code above do?**

Packages scaled dot-product attention into a reusable module. Trace the shapes through `forward`, and notice the **flatten → attend → un-flatten** skeleton from the deep dive:

| Line | Operation | Shape after |
|------|-----------|-------------|
| `inp = x` | stash for residual | `(64,32,16,16)` |
| `self.norm(x)` | GroupNorm pre-norm | `(64,32,16,16)` |
| `.view(n,c,-1).transpose(1,2)` | flatten grid → sequence | `(64,256,32)` |
| `self.q/k/v(x)` | Q, K, V projections | `(64,256,32)` each |
| `q@k.transpose(1,2)/scale` | scaled scores | `(64,256,256)` |
| `s.softmax(dim=-1)@v` | weights, then mix values | `(64,256,32)` |
| `self.proj(x)` | output projection | `(64,256,32)` |
| `.transpose(1,2).reshape(n,c,h,w)` | un-flatten → image | `(64,32,16,16)` |
| `x + inp` | residual add | `(64,32,16,16)` |

Two design choices worth flagging:

- **Pre-norm.** `self.norm` is applied to the *input* before projections. `nn.GroupNorm(1, ni)` — GroupNorm with a **single group** — normalizes across *all* channels and spatial positions of each image, which is equivalent to LayerNorm over `(C,H,W)`. This stabilizes the scale of what enters the attention, a standard transformer-style pre-norm.
- **Residual.** `return x + inp` means the module learns a *correction* to add to its input rather than a full replacement. If attention has nothing useful to add, it can learn to output ≈ 0 and the block becomes an identity — the same "make the layer easy to skip" logic as ResNet blocks. It also keeps output statistics near the input's (we'll verify the std stays ≈ 1 shortly).


In [ ]:
sa = SelfAttention(32)   # instantiate for a 32-channel feature map

**What does the code above do?**

Builds a `SelfAttention` module sized for 32 channels. Its Q/K/V/proj weights are randomly initialized; we are only checking that shapes and numerics flow correctly, not training anything.


In [ ]:
ra = sa(x)      # run our dummy activation through attention
ra.shape

torch.Size([64, 32, 16, 16])

**What does the code above do?**

Runs `x` through the module. The output shape is **identical to the input**, `(64, 32, 16, 16)` — exactly what we need for a drop-in block: attention consumes a feature map and returns a feature map of the same shape, so it can be inserted anywhere in a conv network (e.g. between U-Net blocks) without disturbing the surrounding plumbing.


In [ ]:
ra[0,0,0]      # peek at one row of one channel of the first image's output

tensor([ 0.9542,  0.9744, -1.1990, -0.0605,  0.6420,  1.2530,  0.9818, -0.8887,
        -0.7011,  0.4919,  0.6624, -0.0641,  0.7628, -0.0771, -0.5991, -0.7150],
       grad_fn=<SelectBackward0>)

**What does the code above do?**

Prints one 16-long row of the output (image 0, channel 0, row 0) — 16 numbers, one per column of the feature map. There is nothing special about the values themselves; we grab them so that, in the next section, we can compare them against HuggingFace's `AttentionBlock` output and see that the two match. (`grad_fn=<SelectBackward0>` just confirms the tensor is still part of the autograd graph.)


---

# Part 5: Validating Against 🤗 diffusers' `AttentionBlock`

Talk is cheap — let's *prove* our implementation is correct. Strategy: take HuggingFace's production `AttentionBlock`, **copy our exact weights into it**, run the same input through both, and check the outputs are numerically identical. If they are, our from-scratch code implements the same function.

---


In [ ]:
def cp_parms(a,b):
    # Copy the learnable parameters of layer a into layer b.
    # We assign the SAME Parameter objects, so a and b share weights/bias.
    b.weight = a.weight   # copy the weight matrix
    b.bias = a.bias       # copy the bias vector

**What does the code above do?**

A tiny helper that copies the `weight` and `bias` of one layer into another. By assigning the *same* `Parameter` objects, layer `b` ends up using exactly `a`'s learned values — the mechanism we need to force HuggingFace's block to compute with *our* randomly-initialized weights instead of its own. (This works because both our `nn.Linear`s and diffusers' internal projections store their parameters under the standard `.weight`/`.bias` attribute names.)


In [ ]:
# =====================================================
# BUILD HF BLOCK AND TRANSPLANT OUR WEIGHTS INTO IT
# =====================================================
at = AttentionBlock(32, norm_num_groups=1)   # HF block, matching our GroupNorm(1, .) choice
# Pair up OUR sublayers with the CORRESPONDING sublayers in HF block, then copy each ours -> theirs:
src = sa.q,sa.k,sa.v,sa.proj,sa.norm
dst = at.query,at.key,at.value,at.proj_attn,at.group_norm
for s,d in zip(src,dst): cp_parms(s,d)

**What does the code above do?**

Instantiates HuggingFace's `AttentionBlock(32, norm_num_groups=1)` — the `norm_num_groups=1` makes *their* normalization match *our* `GroupNorm(1, 32)` — and then transplants our five learnable sublayers into their corresponding slots:

| Ours | HuggingFace's name |
|------|--------------------|
| `sa.q` | `at.query` |
| `sa.k` | `at.key` |
| `sa.v` | `at.value` |
| `sa.proj` | `at.proj_attn` |
| `sa.norm` | `at.group_norm` |

After this loop, `at` uses **our** weights everywhere. The naming differences (`q` vs `query`, `proj` vs `proj_attn`) are cosmetic — the underlying math is the same, which is the entire point we are about to confirm.


In [ ]:
rb = at(x)     # run the SAME input through HF's block (now using our weights)
rb[0,0,0]

tensor([ 0.9542,  0.9744, -1.1990, -0.0605,  0.6420,  1.2530,  0.9818, -0.8887,
        -0.7011,  0.4919,  0.6624, -0.0641,  0.7628, -0.0771, -0.5991, -0.7150],
       grad_fn=<SelectBackward0>)

**What does the code above do?**

Runs the identical input `x` through HuggingFace's block and prints the same slice. **Compare it to `ra[0,0,0]` from Part 4 — the numbers are identical.** That is the proof: given the same weights and the same input, our hand-written `SelfAttention` and HuggingFace's production `AttentionBlock` compute exactly the same function. Our from-scratch implementation is correct.

This "transplant weights and diff the outputs" technique is a fantastic general debugging tactic — any time you reimplement a published layer, borrow the reference implementation and check for bit-for-bit (or float-tolerance) agreement rather than eyeballing loss curves.


---

# Part 6: One Projection Instead of Three — the Combined QKV

Three separate `nn.Linear(32, 32)` layers for Q, K, V is a little wasteful: they all read the same input, so we can fuse them into a **single** `nn.Linear(32, 96)` that outputs all three stacked, then split the result back apart with `torch.chunk`. Same math, one matmul, cleaner code.

---


In [ ]:
# =====================================================
# ONE PROJECTION THAT PRODUCES Q, K, V STACKED
# =====================================================
sqkv = nn.Linear(ni, ni*3)   # 32 -> 96 : outputs Q||K||V concatenated along the feature axis
st = sqkv(t)                 # apply to the sequence t : (64, 256, 32) -> (64, 256, 96)
st.shape

torch.Size([64, 256, 96])

**What does the code above do?**

Replaces the three `32→32` projections with one `32→96` projection (`ni*3 = 96`). Applied to `t`, it yields `(64, 256, 96)`: at each position we now have a 96-dimensional vector that is the query (first 32), key (next 32), and value (last 32) concatenated. One matmul does the work of three, and the weight layout is identical to stacking the three separate weight matrices.


In [ ]:
# =====================================================
# SPLIT THE 96 BACK INTO THREE 32-WIDE CHUNKS
# =====================================================
q,k,v = torch.chunk(st, 3, dim=-1)   # split last dim (96) into 3 equal pieces of 32
q.shape

torch.Size([64, 256, 32])

**What does the code above do?**

`torch.chunk(st, 3, dim=-1)` slices the last axis (96) into 3 equal blocks of 32 and returns them as `q, k, v`, each `(64, 256, 32)` — exactly the shapes we had from the three separate layers. So `chunk` is the inverse of the "concatenate the outputs" that the fused `nn.Linear(32, 96)` implicitly did.


In [ ]:
# Sanity check that scores still come out (256, 256) per image with the fused Q/K/V.
# (Here the operands are written k@q^T instead of q@k^T; for a shape check the order is irrelevant.)
(k@q.transpose(1,2)).shape

torch.Size([64, 256, 256])

**What does the code above do?**

Confirms the fused Q/K/V still produce a `(64, 256, 256)` score matrix — identical shape to the three-layer version. (Jeremy writes `k@q.transpose(1,2)` here rather than `q@k.transpose(1,2)`; for a *shape* check the operand order doesn't matter, though for the actual attention values the convention `q@kᵀ` is what row-wise softmax expects.)


### The refactored module (v2): combined QKV + BatchNorm

Now the same class, rebuilt around the fused projection. Note it also swaps the pre-norm from `GroupNorm` to `BatchNorm2d` — a small variation Jeremy tries here.


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, ni):
        super().__init__()
        self.scale = math.sqrt(ni)
        self.norm = nn.BatchNorm2d(ni)      # this version uses BatchNorm2d as the pre-norm
        self.qkv = nn.Linear(ni, ni*3)      # ONE fused projection: 32 -> 96
        self.proj = nn.Linear(ni, ni)       # output projection

    def forward(self, inp):
        n,c,h,w = inp.shape
        x = self.norm(inp).view(n, c, -1).transpose(1, 2)   # norm, then flatten -> (n, S, C)
        q,k,v = torch.chunk(self.qkv(x), 3, dim=-1)         # one matmul, split into Q,K,V
        s = (q@k.transpose(1,2))/self.scale                 # scaled scores (n, S, S)
        x = s.softmax(dim=-1)@v                             # mix values     (n, S, C)
        x = self.proj(x).transpose(1,2).reshape(n,c,h,w)    # project + un-flatten -> image
        return x+inp                                        # residual

**What does the code above do?**

The same scaled dot-product attention as v1, with two cosmetic changes:

1. **Fused QKV.** `self.qkv = nn.Linear(ni, ni*3)` plus `torch.chunk(..., 3, dim=-1)` replaces the three separate projections. Fewer parameters *objects* (though the same parameter *count*), one matmul, tidier code.
2. **BatchNorm pre-norm.** `nn.BatchNorm2d(ni)` in place of `GroupNorm(1, ni)`. Applied to the 4D input before flattening, it normalizes each channel across the batch and spatial dims. (In practice GroupNorm is usually preferred inside generative U-Nets because it doesn't depend on batch statistics, but this shows the block is agnostic to the choice of pre-norm.)

The **flatten → attend → un-flatten → residual** skeleton is unchanged.


### A cautionary redefinition (v3): what happens if you *drop* the flatten

Jeremy immediately writes the class **once more**, stripped down further — removing the residual, the reshape-back-to-image, *and* (critically) the `.view(n,c,-1)` flatten step. This one is instructive precisely because **it does not run**. Preserving it exactly:


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, ni):
        super().__init__()
        self.scale = math.sqrt(ni)
        self.norm = nn.BatchNorm2d(ni)
        self.qkv = nn.Linear(ni, ni*3)
        self.proj = nn.Linear(ni, ni)

    def forward(self, x):
        x = self.norm(x).transpose(1, 2)   # NOTE: no .view(n,c,-1) flatten before transpose!
        q,k,v = torch.chunk(self.qkv(x), 3, dim=-1)
        s = (q@k.transpose(1,2))/self.scale
        x = s.softmax(dim=-1)@v
        return self.proj(x).transpose(1,2)

**What does the code above do? (and why it's here)**

This is a **broken, experimental redefinition** — a scratch version where Jeremy is exploring how minimal the block can get (no residual, no un-flatten). The fatal difference from v2 is the missing flatten:

```python
x = self.norm(x).transpose(1, 2)          # v3: 4D (n,c,h,w) -> transpose -> (n,h,c,w)   ✗
x = self.norm(inp).view(n,c,-1).transpose(1,2)  # v2: flatten to (n,c,S) FIRST, then -> (n,S,c)  ✓
```

With a 4D input `(64, 32, 16, 16)`, `.transpose(1, 2)` swaps the channel and height axes to `(64, 16, 32, 16)` — leaving **16** (a spatial axis), not **32** (the channels), as the last dimension. The very next line, `self.qkv(x)`, is `nn.Linear(32, 96)` and demands a last dimension of 32. It gets 16 and raises:

```
RuntimeError: mat1 and mat2 shapes cannot be multiplied (32768x16 and 32x96)
```

**This is the payoff of the Part-2 deep dive.** The `.view(n,c,-1)` flatten is not decorative bookkeeping — it is what puts the *channel* dimension last so `nn.Linear` and the attention matmuls line up. Drop it and attention doesn't just misbehave, it fails to run at all.

So how did the sanity-check outputs below (a valid shape, `std ≈ 1.0`) get produced? Because in the saved notebook they were computed while the **working v2** definition was the live one — the outputs belong to v2, not to this broken v3. We keep v3 here exactly as Jeremy left it, as a worked example of the failure mode.


In [ ]:
sa = SelfAttention(32)
sa(x).shape

torch.Size([64, 32, 16, 16])

**What does the code above do?**

Instantiates `SelfAttention(32)` and runs `x` through it, reporting `torch.Size([64, 32, 16, 16])`. As explained above, this saved output corresponds to the **working v2** module (which restores the image shape and adds the residual). If you run this cell *after* executing the broken v3 definition, you will instead get the `RuntimeError` shown above — a useful thing to try yourself to cement why the flatten matters.


In [ ]:
sa(x).std()

tensor(1.0047, grad_fn=<StdBackward0>)

**What does the code above do?**

Reports the standard deviation of the output: **≈ 1.0**. This is a health check. A well-behaved block should preserve the scale of activations — if attention blew the variance up or crushed it toward 0, deep stacks of these blocks would explode or vanish. The std stays near 1 here largely thanks to the **residual add** (the output is *input + small correction*), confirming the block is scale-preserving. (This value, too, is from the working v2 definition.)


---

# Part 7: Multi-Head Attention

So far there is a **single** attention operation mixing all 32 channels at once. **Multi-head attention** splits the feature dimension into several independent *heads* (say 4 heads of 8 channels each), runs attention separately within each head, then concatenates the results.

**Why bother?** A single attention map forces every channel to agree on *one* `256×256` pattern of "who attends to whom." Multiple heads let the network learn *several different* attention patterns simultaneously — one head might track long-range spatial layout, another local texture, another color coherence — and combine them. It is the same "let different subspaces specialize" intuition as having multiple conv filters, applied to the attention pattern itself.

We'll first build the reshaping by hand, then see the `einops` one-liner that replaces it.

---


In [ ]:
# =====================================================
# MANUAL HEAD-SPLITTING RESHAPES (the verbose way)
# =====================================================
def heads_to_batch(x, heads):
    # Fold the `heads` sub-vectors of each feature into the BATCH axis, so a plain
    # (batched) attention can process all heads independently in parallel.
    n,sl,d = x.shape                       # (batch, seq_len, full_feature_dim)
    x = x.reshape(n, sl, heads, -1)        # split features -> (n, sl, heads, d/heads)
    return x.transpose(2, 1).reshape(n*heads,sl,-1)   # -> (n*heads, sl, d/heads)

def batch_to_heads(x, heads):
    # The inverse: pull the heads back out of the batch axis and re-concatenate them
    # along the feature axis.
    n,sl,d = x.shape                       # here n is really n*heads
    x = x.reshape(-1, heads, sl, d)        # (n, heads, sl, d)
    return x.transpose(2, 1).reshape(-1,sl,d*heads)   # -> (n, sl, d*heads)

**What does the code above do?**

Two helper functions that convert between "one big feature vector" and "several per-head feature vectors", using the **batch axis** as a parking spot for the heads:

- **`heads_to_batch`**: takes `(n, seq, d)`, splits the `d` features into `heads` groups of `d/heads`, and *moves the head axis into the batch* to produce `(n·heads, seq, d/heads)`. Now an ordinary batched attention treats each head as just another independent item in the batch — no special-case code needed.
- **`batch_to_heads`**: the exact inverse — pulls the heads back out of the batch and concatenates them along the feature axis, restoring `(n, seq, d)`.

The `transpose` in the middle of each is what actually reorders head vs. sequence so the final `reshape` groups the right elements. These work, but they are fiddly and easy to get subtly wrong — which motivates the next cell.


In [ ]:
from einops import rearrange   # a readable, index-named tensor reshaping DSL

**What does the code above do?**

Imports `rearrange` from **einops**. einops lets you describe a reshape/transpose with *named axes* in a string pattern instead of a sequence of `.reshape`/`.transpose` calls. It is far less error-prone: the pattern documents itself, and einops checks that your axis arithmetic is consistent.


In [ ]:
# einops replaces BOTH helper functions with a single readable pattern.
# 'n s (h d) -> (n h) s d' reads as: the feature axis is (heads * head_dim); pull heads
# out and fold them into the batch. h=8 tells einops how many heads to split into.
t2 = rearrange(t , 'n s (h d) -> (n h) s d', h=8)
t.shape, t2.shape

(torch.Size([64, 256, 32]), torch.Size([512, 256, 4]))

**What does the code above do?**

This one line does exactly what `heads_to_batch` did. Read the pattern `'n s (h d) -> (n h) s d'`:

- **Left of `->`** names the input axes: `n` (batch=64), `s` (seq=256), and `(h d)` — it declares that the last axis (32) *factors* into `h` heads times `d` per-head features. With `h=8`, einops infers `d = 32/8 = 4`.
- **Right of `->`** is the desired layout: `(n h)` fuses batch and heads into one axis of size `64·8 = 512`, `s` stays, and `d = 4` is the new per-head feature size.

Result: `(64, 256, 32) → (512, 256, 4)`. The 8 heads now ride in the batch axis, each carrying its own 4-dim slice of the features. Compare the readability to the two-function manual version — this is why einops is standard in modern attention code.


In [ ]:
# The inverse pattern reassembles the heads back onto the feature axis.
t3 = rearrange(t2, '(n h) s d -> n s (h d)', h=8)

In [ ]:
t2.shape,t3.shape

(torch.Size([512, 256, 4]), torch.Size([64, 256, 32]))

**What does the code above do?**

Runs the reverse pattern `'(n h) s d -> n s (h d)'` (einops equivalent of `batch_to_heads`): it splits the fused `512` batch axis back into `n=64` and `h=8`, then folds the heads back into the feature axis to restore `(64, 256, 32)`. The two prints confirm `t2` is the split form `(512, 256, 4)` and `t3` is back to the original `(64, 256, 32)`.


In [ ]:
(t==t3).all()   # round-trip check: split then rejoin must recover the original exactly

tensor(True)

**What does the code above do?**

The crucial correctness check: after splitting into heads and rejoining, do we get **exactly** the original tensor back? `(t == t3).all()` returns `tensor(True)`, so the round trip is lossless — einops' factor-and-fuse is a pure re-view of the same data, no elements scrambled. Confidence established, we can safely use `rearrange` inside the real module.


## Deep Dive: Multi-Head Attention and the `(n h) s d` Trick

Multi-head attention has two ideas worth separating cleanly: *why* multiple heads help, and *how* the einops reshape implements them essentially for free.

### Why multiple heads

With a single head over `d = 32` channels, the attention weights `A` are one `256×256` matrix per image. Every channel's value-mixing is governed by that *same* routing. That is a real limitation: the relationship that best explains "spatial layout" may be different from the one that best explains "local texture," yet a single head must compromise on one shared pattern.

Multi-head attention gives each head its **own** Q/K/V subspace (here `32/4 = 8` or `32/8 = 4` dims) and therefore its **own** `256×256` attention matrix. Heads run fully in parallel and their outputs are concatenated, so the layer can attend along several relational axes at once and let the output projection blend them. The total parameter and FLOP cost is essentially unchanged — you have partitioned the same `d` channels into groups, not added new ones.

### Why "fold heads into the batch" is the elegant implementation

The naive implementation of `h` heads is a Python loop: slice out each head's channels, run attention, stack the results. That is slow and ugly. The trick is to notice that **each head's attention is completely independent of the others** — exactly like different *images* in a batch are independent. So if we can make the heads *look like extra batch items*, a single ordinary batched attention handles all of them at once, on the GPU, with zero special-case code.

That is precisely what `'n s (h d) -> (n h) s d'` does:

```
   (n, s,  d)            factor d = h*head_dim,        (n*h, s, head_dim)
  (64,256, 32)   ────────  move h next to n  ───────▶  (512, 256, 4)
                                                        heads now ride in the batch axis
```

Now `q@k.transpose(1,2)` produces `(512, 256, 256)` — that is 512 = 64×8 independent `256×256` attention matrices, one per (image, head) pair, computed in a single batched matmul. After mixing values, the reverse pattern `'(n h) s d -> n s (h d)'` re-concatenates the heads back into a full `32`-dim feature per position. **No loop, no gather, just two re-views of the same memory.**

### Why the scale changes to `√(d/heads)`

Recall from the scaled-dot-product deep dive that the `√d` divisor exists to counteract the variance-`d` growth of a dot product over `d` terms. In multi-head attention each dot product runs over only `d/heads` terms (each head is smaller), so the right normalizer is `√(d/heads)`, not `√d`. That is exactly why the multi-head module below sets:

```python
self.scale = math.sqrt(ni/nheads)   # per-head feature dim is ni/nheads
```

Using the full `√ni` here would over-shrink the scores (softmax too flat); using nothing would let them grow (softmax too peaked). `√(per-head dim)` is the matched choice.


In [ ]:
class SelfAttentionMultiHead(nn.Module):
    def __init__(self, ni, nheads):
        super().__init__()
        self.nheads = nheads
        self.scale = math.sqrt(ni/nheads)   # NOTE: per-head dim -> sqrt(ni/nheads), not sqrt(ni)
        self.norm = nn.BatchNorm2d(ni)
        self.qkv = nn.Linear(ni, ni*3)      # fused Q/K/V as before
        self.proj = nn.Linear(ni, ni)

    def forward(self, inp):
        n,c,h,w = inp.shape
        x = self.norm(inp).view(n, c, -1).transpose(1, 2)      # -> (n, S, C)
        x = self.qkv(x)                                        # -> (n, S, 3C)
        x = rearrange(x, 'n s (h d) -> (n h) s d', h=self.nheads)  # fold heads into batch: (n*heads, S, 3C/heads)
        q,k,v = torch.chunk(x, 3, dim=-1)                      # split Q,K,V (per head): each (n*heads, S, C/heads)
        s = (q@k.transpose(1,2))/self.scale                    # per-head scores: (n*heads, S, S)
        x = s.softmax(dim=-1)@v                                # per-head value mix: (n*heads, S, C/heads)
        x = rearrange(x, '(n h) s d -> n s (h d)', h=self.nheads)  # un-fold heads back onto features: (n, S, C)
        x = self.proj(x).transpose(1,2).reshape(n,c,h,w)       # project + un-flatten -> image
        return x+inp                                           # residual

**What does the code above do?**

The full multi-head module. It is the fused-QKV v2 with **two `rearrange` calls wrapped around the attention core**:

1. Flatten to a sequence and fuse-project as before → `(n, S, 3C)`.
2. **`rearrange('n s (h d) -> (n h) s d')`** folds the heads into the batch axis. Everything after this line is *ordinary single-head attention* — it simply runs on `n·heads` "images" instead of `n`.
3. `chunk` into Q/K/V, compute scaled scores with the per-head `√(ni/nheads)` scale, softmax, mix values — all batched over `(image × head)`.
4. **`rearrange('(n h) s d -> n s (h d)')`** pulls the heads back out and concatenates them into the full feature vector.
5. Output projection, un-flatten to `(n,c,h,w)`, residual add.

The beauty is that steps 2 and 4 are the *only* additions versus single-head attention — the expensive middle is untouched, yet the layer now supports as many heads as you like. `self.proj` at the end is doing double duty: remapping value-space back to feature-space *and* learning how to blend the concatenated heads.


In [ ]:
sa = SelfAttentionMultiHead(32, 4)   # 4 heads of 32/4 = 8 channels each
sx = sa(x)
sx.shape

torch.Size([64, 32, 16, 16])

**What does the code above do?**

Builds a 4-head attention over 32 channels (8 channels per head) and runs `x` through it. Output shape is `(64, 32, 16, 16)` — same as the input, same as single-head. From the *outside*, a multi-head block is a drop-in replacement for a single-head one; the head machinery is entirely internal.


In [ ]:
sx.mean(),sx.std()

(tensor(0.0248, grad_fn=<MeanBackward0>),
 tensor(1.0069, grad_fn=<StdBackward0>))

**What does the code above do?**

Health check again: output mean ≈ 0.025, std ≈ 1.007. Both are close to their inputs' statistics (mean 0, std 1), so the multi-head block is scale-preserving too — safe to stack. The residual connection is again what keeps these numbers anchored near the input distribution.


---

# Part 8: Sanity Check vs. PyTorch's `nn.MultiheadAttention`

PyTorch ships a built-in `nn.MultiheadAttention`. It is worth running it on the same sequence to confirm our understanding of the interface and to see that a standard, well-initialized multi-head attention also yields nicely-scaled outputs.

---


In [ ]:
# =====================================================
# PYTORCH'S BUILT-IN MULTI-HEAD ATTENTION
# =====================================================
nm = nn.MultiheadAttention(32, num_heads=8, batch_first=True)  # 32 feats, 8 heads, (batch, seq, feat) layout
# Self-attention = pass the SAME sequence as query, key, and value:
nmx,nmw = nm(t,t,t)      # nmx: attended output (64,256,32);  nmw: the attention weight maps
nmx = nmx+t              # add our own residual (nn.MultiheadAttention does NOT add one internally)

**What does the code above do?**

Runs PyTorch's reference multi-head attention on our sequence `t`:

- **`nn.MultiheadAttention(32, num_heads=8, batch_first=True)`** — 32-dim features, 8 heads, and `batch_first=True` so it expects `(batch, seq, feat)` (matching our `t`; the default is the transposed `(seq, batch, feat)`, a classic footgun).
- **`nm(t, t, t)`** — passing the same tensor as query, key, and value is what makes it *self*-attention (as opposed to cross-attention, where query comes from one source and key/value from another — the mechanism the conditioned diffusion U-Net will use later). It returns both the output `nmx` and the attention weights `nmw`.
- **`nmx = nmx + t`** — we add the residual ourselves, because `nn.MultiheadAttention` computes only the attention operation and leaves norm/residual/projection-wrapping to the caller (unlike our all-in-one `SelfAttention` modules).


In [ ]:
nmx.mean(),nmx.std()

(tensor(-0.0021, grad_fn=<MeanBackward0>),
 tensor(1.0015, grad_fn=<StdBackward0>))

**What does the code above do?**

Reports the built-in's output statistics after the residual: mean ≈ 0, std ≈ 1.00 — essentially identical to our from-scratch `SelfAttentionMultiHead`. This closes the loop: our hand-built multi-head attention behaves like PyTorch's production implementation, just as our single-head block matched HuggingFace's `AttentionBlock`. We have re-derived, from scratch, exactly the operation the big libraries ship.


---

# Summary and What's Next

### What we built

| Step | Object | Key idea |
|------|--------|----------|
| Reshape | `x.view(n,c,-1).transpose(1,2)` | Treat an image's `H·W` positions as a **sequence** of `C`-dim tokens |
| Core math | `softmax(q@kᵀ/√d) @ v` | Content-based, global, dynamically-weighted mixing of values |
| v1 module | `SelfAttention` (3× Linear, GroupNorm) | Package as a residual block that returns a same-shape feature map |
| Validation | copy weights into 🤗 `AttentionBlock` | Bit-for-bit match ⇒ our implementation is correct |
| v2 module | fused `nn.Linear(ni, 3ni)` + `chunk` | One matmul for Q, K, V |
| Multi-head | `rearrange('n s (h d) -> (n h) s d')` | Fold heads into the batch → several attention patterns for free |
| Cross-check | `nn.MultiheadAttention` | Matches PyTorch's built-in behavior |

### The three ideas to remember

1. **The reshape is the trick.** Attention is a sequence operation; images become sequences by flattening `H×W` into a length-`H·W` sequence with channels last. Flatten → attend → un-flatten. Dropping the flatten doesn't just misbehave — it fails to run (Part 6, v3).
2. **`softmax(QKᵀ/√d)V` is the entire mechanism.** `QKᵀ` builds an all-pairs affinity matrix (the "who couples to whom" / teleconnection matrix), `/√d` keeps softmax well-conditioned, softmax turns affinities into per-row attention weights, and `@V` mixes values accordingly. Everything else is packaging.
3. **Multi-head = fold heads into the batch.** Because heads are mutually independent, `rearrange` parks them in the batch axis and one batched attention does them all, giving the network several simultaneous attention patterns at essentially no extra cost.

### Where this goes

This `SelfAttention`/`SelfAttentionMultiHead` block is a **drop-in module for the diffusion U-Net**. In the notebooks that follow, attention gets inserted at the low-resolution (deepest) stages of the U-Net, where a global "does this whole image cohere?" operation is most valuable and where the `S = H·W` sequence length is small enough to afford the `O(S²)` attention matrix. The cross-attention variant (query from the image, key/value from a conditioning signal like a text embedding or a class label) is what turns an unconditional generator into a *conditioned* one — same math as here, with the key/value sequence coming from somewhere else.

**Climate/EO bridge, revisited.** The `256×256` attention matrix you built is a learned, data-dependent spatial-coupling operator — the deep-learning cousin of a teleconnection/covariance matrix. Where a convolution can only express local coupling and an EOF gives you *fixed* global modes, attention learns *input-specific* long-range coupling on the fly. That is a genuinely useful lens if you ever apply these blocks to spatial climate fields (downscaling, emulation, infilling): the attention map is inspectable, and it is telling you which locations the model thinks are dynamically linked for *this* field.

### Suggested next steps

1. Re-read the two deep dives (scaled dot-product; multi-head `(n h) s d`) — those are where the concepts (and any of my errors) concentrate. Push back on anything that doesn't match your understanding.
2. Try the "break it yourself" exercise: run the Part-6 v3 definition and then `sa(x)` to see the shape error firsthand, then add the `.view(n,c,-1)` back and watch it recover.
3. When satisfied, run `concept-extraction` on this notebook to mint cards (candidates: the flatten reshape, `softmax(QKᵀ/√d)V`, the `√d` scaling derivation, Q vs K vs V roles, the fold-heads-into-batch trick, pre-norm + residual packaging).
4. Optionally `/colab` for a GPU-ready version and `/html` to publish the styled page.

---
